In [ ]:
import sys as _sys
from pathlib import Path as _ResearchPath
_sys.path.insert(0,str(_ResearchPath('.').resolve()/'src'))
from lppls_research import TemporalAvailabilityValidator
import pandas as pd

In [ ]:
df = pd.read_csv(
    "lppls_rolling_results.csv",
    parse_dates=["win_start", "win_end"]
)
df.head()

In [ ]:
data_union = [
    ["UNION01", "2016-06-08", "2016-06-18"],
    ["UNION02", "2016-12-30", "2017-01-15"],
    ["UNION03", "2017-02-22", "2017-03-07"],
    ["UNION04", "2017-04-27", "2017-07-05"],
    ["UNION05", "2017-07-15", "2018-01-09"],
    ["UNION06", "2019-06-17", "2019-07-13"],
    ["UNION07", "2019-08-09", "2019-08-10"],
    ["UNION08", "2019-12-06", "2020-01-05"],
    ["UNION09", "2020-02-07", "2020-02-17"],
    ["UNION10", "2020-03-20", "2020-09-21"],
    ["UNION11", "2020-11-06", "2021-05-20"],
    ["UNION12", "2021-08-08", "2021-09-19"],
    ["UNION13", "2021-10-05", "2021-12-02"],
    ["UNION14", "2024-03-04", "2024-03-25"],
]

bubbles = pd.DataFrame(
    data_union,
    columns=["label_id", "start_date", "end_date"]
)
bubbles["start_date"] = pd.to_datetime(bubbles["start_date"])
bubbles["end_date"] = pd.to_datetime(bubbles["end_date"])

bubbles.head()

In [ ]:
# 3. Utworzenie nowych kolumn

df["literature_bubble"] = 0
df["bubble_id"] = pd.NA
df["bubble_overlap_days"] = 0

window_days = (df["win_end"] - df["win_start"]).dt.days + 1


# 4. Etykietowanie okien LPPLS

for _, bubble in bubbles.iterrows():

    start = bubble["start_date"]
    end = bubble["end_date"]
    label = bubble["label_id"]

    # Czy dzień końca okna przypada w bańce?
    inside = df["win_end"].between(start, end)

    df.loc[inside, "literature_bubble"] = 1
    df.loc[inside, "bubble_id"] = label

    # Ile dni okna pokrywa się z bańką?
    overlap_start = df["win_start"].clip(lower=start)
    overlap_end = df["win_end"].clip(upper=end)

    overlap_days = (
        (overlap_end - overlap_start)
        .dt.days
        .add(1)
        .clip(lower=0)
    )

    df["bubble_overlap_days"] += overlap_days


# 5. Procent pokrycia okna przez bańki

df["bubble_overlap_ratio"] = (
    df["bubble_overlap_days"] / window_days
)


In [ ]:
output_file = "lppls_rolling_results_z_etykietami_baniek.csv"

df.to_csv(output_file, index=False)


In [ ]:
print("Liczba wszystkich okien:", len(df))

print("\nRozkład etykiet:")
print(df["literature_bubble"].value_counts())

print("\nLiczba okien w poszczególnych bańkach:")
print(df["bubble_id"].value_counts().sort_index())

print("\nZapisano:", output_file)
